In [5]:
from experiment_analysis import load_experiment_latencies

# Load experiment 3 from ./experiments/3
df = load_experiment_latencies(exp_id=1)

df.head()
df.describe(percentiles=[0.5, 0.9, 0.99])[[
    "end_to_end_s",
    "server_roundtrip_s",
    "router_queue_s",
    "vllm_compute_s",
]]


,end_to_end_s,server_roundtrip_s,router_queue_s,vllm_compute_s
count,200.000000,200.000000,200.000000,200.000000
mean,63.411753,60.006660,0.462031,26.845720
std,24.302310,22.946210,0.082548,23.968466
min,12.458647,12.067640,0.244588,1.347557
50%,64.646775,61.528780,0.461315,19.684447
90%,90.365901,86.450117,0.583740,64.425536
99%,110.098940,104.867646,0.638379,97.062517
max,132.889515,127.257348,0.640009,101.486898


In [6]:
df

,idx,req_id,send_failed,wait_wall_s,model_latency_s,finish_reason,end_to_end_s,client_to_router_s,router_queue_s,router_to_sidecar_s,sidecar_queue_s,vllm_compute_s,sidecar_post_s,sidecar_to_router_s,router_post_result_s,server_roundtrip_s
0,12,4e261e84b95142f8a5b055f6785142a5,False,12.460782,None,None,12.458647,0.391007,0.409432,0.347982,0.002049,11.304140,0.000057,0.003431,0.000512,12.067640
1,3,5465c03d2b9a43d9b9e20cdaea360637,False,13.927733,None,None,13.925802,0.033769,0.333214,0.289926,0.001210,13.264074,0.000059,0.003093,0.000444,13.892034
2,9,d74ef07104e24975ad116ea95b7d4b98,False,17.125880,None,None,17.124322,0.119460,0.431355,0.334788,0.000464,16.233810,0.000054,0.003371,0.001005,17.004862
3,33,ff3417f241534d6faf46aaa280ddff80,False,18.193462,None,None,18.191925,1.020805,0.540304,0.418787,0.001499,13.862412,0.000057,0.003522,2.344485,17.171120
4,42,25fa2318d1d4437d8fdefe591a14dae7,False,18.196927,None,None,18.194342,0.988695,0.542171,0.281019,0.000252,16.377666,0.000088,0.003798,0.000645,17.205647
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,137,05cec7b68d4549e4be037066dd6a7b61,False,108.910437,None,None,108.908558,4.110823,0.417100,0.280828,28.045008,76.051568,0.000154,0.002715,0.000350,104.797735
196,119,2e60ac2415a442f89f33ab440363ec68,False,109.070359,None,None,109.068866,4.493789,0.584073,0.334850,35.386025,68.267131,0.000135,0.002510,0.000346,104.575077
197,189,b0edc2dd6ca64bdf9349c70a6a61a358,False,110.030247,None,None,110.028283,6.457383,0.471287,0.180606,77.019754,25.895167,0.000215,0.003396,0.000470,103.570900
198,94,e5636862c26f41d8ac73fbc0e336882c,False,117.095966,None,None,117.094000,5.305142,0.392438,0.351890,43.053442,67.986767,0.000195,0.003571,0.000546,111.788858


In [8]:
from experiment_analysis import load_experiment_latencies

# -------- choose the two experiments to compare --------
exp_a = 1
exp_b = 3
# --------------------------------------------------------

# Load both experiments
df_a = load_experiment_latencies(exp_id=exp_a)
df_b = load_experiment_latencies(exp_id=exp_b)

# Define which latency columns to compare
cols = [
    "end_to_end_s",
    "server_roundtrip_s",
    "router_queue_s",
    "vllm_compute_s",
]

# Compute percentile summaries for each
summary_a = df_a.describe(percentiles=[0.5, 0.9, 0.99])[cols]
summary_b = df_b.describe(percentiles=[0.5, 0.9, 0.99])[cols]

# Rename columns to tag experiment ID
summary_a = summary_a.add_prefix(f"exp{exp_a}_")
summary_b = summary_b.add_prefix(f"exp{exp_b}_")

# Combine side-by-side
comparison = summary_a.join(summary_b)

comparison


,exp1_end_to_end_s,exp1_server_roundtrip_s,exp1_router_queue_s,exp1_vllm_compute_s,exp3_end_to_end_s,exp3_server_roundtrip_s,exp3_router_queue_s,exp3_vllm_compute_s
count,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000
mean,63.411753,60.006660,0.462031,26.845720,66.462851,64.679704,30.563676,28.263106
std,24.302310,22.946210,0.082548,23.968466,24.293512,23.635929,24.358065,24.876630
min,12.458647,12.067640,0.244588,1.347557,17.846678,17.316734,0.053590,0.525727
50%,64.646775,61.528780,0.461315,19.684447,74.541749,72.198223,31.693114,21.130279
90%,90.365901,86.450117,0.583740,64.425536,91.265922,89.400967,63.991271,70.832426
99%,110.098940,104.867646,0.638379,97.062517,114.761590,112.619323,71.410790,90.124729
max,132.889515,127.257348,0.640009,101.486898,124.252173,122.677347,72.468750,107.447617


In [9]:
from experiment_analysis import load_experiment_latencies
import matplotlib.pyplot as plt

# ---- choose experiment ----
exp_id = 1
# ---------------------------

# Load and order by request index (or any monotonic field you prefer)
df = load_experiment_latencies(exp_id=exp_id).sort_values("idx").reset_index(drop=True)

# X-axis: request index (0, 1, 2, ...)
x = df["idx"]

# Latency metrics to plot over time
latency_cols = [
    "end_to_end_s",
    "server_roundtrip_s",
    "router_queue_s",
    "router_to_sidecar_s",
    "sidecar_queue_s",
    "vllm_compute_s",
]

plt.figure(figsize=(12, 6))

for col in latency_cols:
    if col in df.columns:
        plt.plot(x, df[col], label=col)

plt.xlabel("Request index")
plt.ylabel("Latency (seconds)")
plt.title(f"Per-request latency components over time (experiment {exp_id})")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()


FileNotFoundError: Experiment directory not found: experiments/1